## Tutorial: running `minato.ravel` at large scale

This tutorial shows a production pattern for running `minato.ravel` on large samples of stars without keeping thousands of per-star scratch directories. It is based on the SDSS-V multiplicity run started on 2026-05-06, where a `Teff_fit > 10000 K` parent sample of 141,795 stars was split into 10,000-star batches.

This is not a small interactive notebook. Treat it as a template for an astro-node or cluster run. Run a smoke batch first, inspect the outputs, then launch full batches in `tmux` or your scheduler.

### What you need

- A star-level manifest with one row per star.
- A way to load every epoch spectrum for each star. In the SDSS production run this was a full-spectrum normalised HDF5 registry.
- Enough CPU workers for parallel stars, and enough CPU devices per worker for NumPyro chains.
- Local write space for batch outputs, logs, compact resume files, and final parquet tables.

The examples below use SDSS-like column names, but the layout is deliberately simple so it can be adapted to another survey.

In [ ]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path

import pandas as pd

### 1. Define the production root and sample manifest

Keep a single production root with stable subdirectories. The SDSS run used this layout:

```text
ravel_full_sample_20260506/
  manifests/
  python_scripts/
  launch_scripts/
  batch_00000/
  batch_00001/
  combined/
```

The parent manifest should be a frozen input. For the SDSS case study, the parent sample was selected from the locked observed SDSS-V sample plus `other_programmes` epochs for the same locked `SDSS_ID` set, requiring registry-covered spectra, finite XP/BOSS `Teff_fit`, and `Teff_fit > 10000 K`.

In [ ]:
production_root = Path("ravel_full_sample_YYYYMMDD").resolve()
manifest_path = production_root / "manifests" / "ravel_parent_manifest.parquet"

manifests_dir = production_root / "manifests"
launch_dir = production_root / "launch_scripts"
combined_dir = production_root / "combined"

for directory in (manifests_dir, launch_dir, combined_dir):
    directory.mkdir(parents=True, exist_ok=True)

A practical manifest contract is:

- `SDSS_ID`: stable star identifier.
- `SPEC_FILE`: list of epoch spectrum names, or a string representation of that list if the runner consumes CSV.
- Optional QC columns such as `n_epochs`, `xp_teff`, `SN_max`, and sample-selection bins.

For large production, keep parquet as the canonical manifest format and write CSV shards only if the batch runner needs CSV input.

In [ ]:
# Inspect the manifest before splitting it.
# parent = pd.read_parquet(manifest_path)
# parent[["SDSS_ID", "SPEC_FILE"]].head()
# parent.shape

### 2. Choose fitting families and MCMC settings

The SDSS production run used four families. Blue, red, and full are stellar-line measurements; Na is kept as a dedicated diagnostic with a narrowed Na window and stricter missing-window handling in the production runner.

In [ ]:
fit_families = {
    "blue": {"lines": [4026, 4102, 4144, 4340, 4388, 4471, 4713], "sb1_method": "prob"},
    "red": {"lines": [5876, 6678, 9015, 9229, 9546], "sb1_method": "prob"},
    "full": {
        "lines": [
            4026, 4102, 4144, 4340, 4388, 4471, 4713,
            4542, 4686, 5412, 5876, 6678, 9015, 9229, 9546,
        ],
        "sb1_method": "prob",
    },
    "na": {"lines": [5890, 5896], "sb1_method": "na"},
}

ravel_settings = {
    "num_warmup": 200,
    "num_samples": 500,
    "num_chains": 4,
    "chain_method": "parallel",
    "max_interp_points": 400,
    "profile": "Gaussian",
    "Hprofile": "Lorentzian",
    "plots": False,
    "cornerplots": False,
    "verbose": False,
    "progress": False,
}

For parallel chains on CPU, set JAX environment variables before importing `minato.ravel`. In the SDSS run each star worker was given 4 JAX CPU devices, and the node ran 48 star workers. Thread counts were pinned to 1 to avoid multiplying BLAS/OpenMP threads inside each worker.

In [ ]:
def configure_worker_environment(n_cpus_per_worker: int = 4) -> None:
    os.environ.setdefault("MINATO_QUIET", "1")
    os.environ.setdefault("OMP_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
    os.environ.setdefault("JAX_ENABLE_X64", "True")
    os.environ.setdefault(
        "XLA_FLAGS",
        f"--xla_force_host_platform_device_count={n_cpus_per_worker}",
    )

# Call this before importing minato.ravel inside each worker process.
# configure_worker_environment(n_cpus_per_worker=4)
# from minato import ravel

### 3. Split the parent manifest into batches

Use batches large enough that launch overhead is negligible but small enough to resume, inspect, and combine easily. The SDSS run used 10,000 stars per batch. The first two completed batches achieved 1.80 and 1.51 stars/s on 48 workers with 4 CPU devices per worker.

In [ ]:
def write_batch_manifests(parent: pd.DataFrame, production_root: Path, batch_size: int = 10_000) -> pd.DataFrame:
    manifests_dir = production_root / "manifests"
    manifests_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    n_batches = math.ceil(len(parent) / batch_size)
    for batch_number in range(n_batches):
        batch_id = f"batch_{batch_number:05d}"
        first = batch_number * batch_size
        batch = parent.iloc[first:first + batch_size].copy()

        batch_dir = production_root / batch_id
        (batch_dir / "logs" / "failures").mkdir(parents=True, exist_ok=True)

        parquet_path = manifests_dir / f"{batch_id}_manifest.parquet"
        csv_path = manifests_dir / f"{batch_id}_manifest.csv"
        batch.to_parquet(parquet_path, index=False)
        batch.to_csv(csv_path, index=False)

        rows.append({
            "batch_id": batch_id,
            "status": "planned",
            "n_stars_planned": len(batch),
            "manifest_path": str(csv_path),
            "output_dir": str(batch_dir),
        })

    index = pd.DataFrame(rows)
    index.to_csv(manifests_dir / "batch_index.csv", index=False)
    return index

# parent = pd.read_parquet(manifest_path)
# batch_index = write_batch_manifests(parent, production_root, batch_size=10_000)
# batch_index.head()

### 4. Run one star in an isolated scratch directory

A robust batch runner should make each star independent: load spectra, run each family, extract compact RV/QC rows, then delete successful scratch output after writing compact results. Failed or partial stars should produce structured failure rows.

The exact loader depends on your data store. The important part is that `SLfit` receives the same kind of spectrum list it receives in the small SB1/SB2 tutorials, while the large-scale runner controls logging, scratch paths, and retries.

In [ ]:
def run_one_star_template(star_row: dict, scratch_root: Path, hdf5_root: Path) -> dict:
    """Template for the per-star unit of work. Adapt the loader and result parsing."""
    configure_worker_environment(n_cpus_per_worker=4)
    from minato import ravel

    sdss_id = str(star_row["SDSS_ID"])
    star_dir = scratch_root / sdss_id
    star_dir.mkdir(parents=True, exist_ok=True)

    # Replace this with your project loader. It should return the spectrum inputs
    # expected by ravel.SLfit, or write temporary spectra below star_dir.
    spec_files = star_row["SPEC_FILE"]
    data_path = str(hdf5_root)

    family_outputs = []
    failures = []
    for family_name, family_cfg in fit_families.items():
        family_dir = star_dir / family_name
        family_dir.mkdir(parents=True, exist_ok=True)
        try:
            ravel.SLfit(
                spec_files,
                data_path=data_path,
                save_path=str(family_dir),
                lines=family_cfg["lines"],
                SB2=False,
                sb1_method=family_cfg["sb1_method"],
                **ravel_settings,
            )
            family_outputs.append({"family": family_name, "status": "ok", "path": str(family_dir)})
        except Exception as exc:  # Keep this structured in production.
            failures.append({"family": family_name, "status": "failed", "error": repr(exc)})

    return {
        "SDSS_ID": sdss_id,
        "status": "ok" if not failures else "partial",
        "families": family_outputs,
        "failures": failures,
    }

### 5. Run a batch with locks, compact resume logs, and provenance

A production runner should avoid shared mutable files across nodes. Each batch writes its own completion log, failure log, parquet tables, and provenance. Use lock files to avoid accidental duplicate launches:

- Create `batch_XXXXX.RUNNING` at start.
- Refuse to start an existing `RUNNING` or `DONE` batch unless you explicitly resume or force it after inspection.
- Write `batch_XXXXX.DONE` only after parquet tables, failure logs, completed-star logs, and provenance are complete.
- Write a compact `star_results_batch_XXXXX.jsonl` while running so interrupted batches can resume missing stars.

In [ ]:
def batch_output_paths(batch_dir: Path, batch_id: str) -> dict[str, Path]:
    return {
        "running": batch_dir / f"{batch_id}.RUNNING",
        "done": batch_dir / f"{batch_id}.DONE",
        "compact_results": batch_dir / f"star_results_{batch_id}.jsonl",
        "completed_stars": batch_dir / f"completed_stars_{batch_id}.jsonl",
        "epoch_qc": batch_dir / f"ravel_epoch_qc_{batch_id}.parquet",
        "star_qc": batch_dir / f"ravel_star_qc_{batch_id}.parquet",
        "failures": batch_dir / f"ravel_failures_{batch_id}.jsonl",
        "provenance": batch_dir / f"provenance_{batch_id}.json",
    }

# In the real runner, append one compact JSON line per completed star:
# with paths["compact_results"].open("a", encoding="utf-8") as handle:
#     handle.write(json.dumps(star_result, sort_keys=True) + "\n")

### 6. Launch long batches outside the notebook

Do not run full production batches inside Jupyter. Write one launch script per batch and start it with `tmux` or a scheduler. The SDSS production launch used these settings:

```bash
export MINATO_QUIET=1
export OMP_NUM_THREADS=1
export OPENBLAS_NUM_THREADS=1
export MKL_NUM_THREADS=1
export NUMEXPR_NUM_THREADS=1
export JAX_ENABLE_X64=True
export XLA_FLAGS='--xla_force_host_platform_device_count=4'
uv run --no-sync python python_scripts/run_batch.py \
  --batch-id batch_00000 \
  --manifest manifests/batch_00000_manifest.csv \
  --output-dir batch_00000 \
  --n-workers 48 \
  --n-cpus-per-worker 4 \
  --num-warmup 200 \
  --num-samples 500 \
  --num-chains 4 \
  --chain-method parallel \
  --max-interp-points 400 \
  --families blue,red,full,na \
  --profile Gaussian \
  --hprofile Lorentzian
```

Start with a 100-300 star smoke manifest using the same fitting settings. Only launch full batches after the smoke run has produced sensible star-level and epoch-level QC tables.

In [ ]:
def launch_command(batch_id: str, production_root: Path) -> str:
    batch_dir = production_root / batch_id
    return f"""tmux new-session -d -s ravel_full_{batch_id} '
cd {production_root.parent} && \
export MINATO_QUIET=1 && \
export OMP_NUM_THREADS=1 && export OPENBLAS_NUM_THREADS=1 && \
export MKL_NUM_THREADS=1 && export NUMEXPR_NUM_THREADS=1 && \
export JAX_ENABLE_X64=True && \
export XLA_FLAGS=--xla_force_host_platform_device_count=4 && \
uv run --no-sync python {production_root}/python_scripts/run_batch.py \
  --batch-id {batch_id} \
  --manifest {production_root}/manifests/{batch_id}_manifest.csv \
  --output-dir {batch_dir} \
  --n-workers 48 --n-cpus-per-worker 4 \
  --num-warmup 200 --num-samples 500 --num-chains 4 \
  --chain-method parallel --max-interp-points 400 \
  --families blue,red,full,na --profile Gaussian --hprofile Lorentzian \
  > {batch_dir}/logs/batch.stdout.log \
  2> {batch_dir}/logs/batch.stderr.log
'"""

# print(launch_command("batch_00000", production_root))

### 7. Monitor throughput and memory

Track progress from batch provenance rather than by scraping terminal output. The first two SDSS batches completed as follows:

In [ ]:
completed_batches = pd.DataFrame([
    {
        "batch_id": "batch_00000",
        "n_stars_completed": 10_000,
        "n_stars_failed": 13,
        "n_failure_rows": 31,
        "n_epoch_rows": 91_630,
        "wall_time_seconds": 5_561.6,
        "max_worker_rss_kb": 8_260_520,
    },
    {
        "batch_id": "batch_00001",
        "n_stars_completed": 10_000,
        "n_stars_failed": 23,
        "n_failure_rows": 57,
        "n_epoch_rows": 109_458,
        "wall_time_seconds": 6_610.0,
        "max_worker_rss_kb": 9_701_936,
    },
])
completed_batches["throughput_stars_per_s"] = (
    completed_batches["n_stars_completed"] / completed_batches["wall_time_seconds"]
)
completed_batches["max_worker_rss_gb"] = completed_batches["max_worker_rss_kb"] / 1024**2
completed_batches

Useful checks while batches are running:

```bash
tmux list-sessions
tmux attach -t ravel_full_batch_00000_YYYYMMDD
tail -n 50 batch_00000/logs/batch.stderr.log
```

If a node is interrupted, inspect the batch directory first. Resume only after confirming that the compact result log and lock state agree with the parquet outputs.

### 8. Combine completed batches

Combine only per-batch parquet tables and JSONL failure logs. Rebuild the global `batch_index.csv` from batch provenance, not from live workers. This avoids concurrent writes when several nodes run at once.

In [ ]:
def combine_completed_batches(production_root: Path, allow_incomplete: bool = False) -> dict:
    batch_dirs = sorted(production_root.glob("batch_*"))
    epoch_frames = []
    star_frames = []
    failures = []
    batch_rows = []

    for batch_dir in batch_dirs:
        batch_id = batch_dir.name
        provenance_path = batch_dir / f"provenance_{batch_id}.json"
        if not provenance_path.exists():
            continue
        provenance = json.loads(provenance_path.read_text())
        if provenance.get("status") != "done" and not allow_incomplete:
            continue

        epoch_path = batch_dir / f"ravel_epoch_qc_{batch_id}.parquet"
        star_path = batch_dir / f"ravel_star_qc_{batch_id}.parquet"
        failure_path = batch_dir / f"ravel_failures_{batch_id}.jsonl"
        if epoch_path.exists() and star_path.exists():
            epoch_frames.append(pd.read_parquet(epoch_path))
            star_frames.append(pd.read_parquet(star_path))
        if failure_path.exists():
            failures.extend(json.loads(line) for line in failure_path.read_text().splitlines() if line.strip())
        batch_rows.append(provenance)

    if not epoch_frames:
        raise RuntimeError("No completed batch tables found")

    combined = production_root / "combined"
    combined.mkdir(exist_ok=True)
    epoch_all = pd.concat(epoch_frames, ignore_index=True)
    star_all = pd.concat(star_frames, ignore_index=True)
    epoch_all.to_parquet(combined / "ravel_epoch_qc_all.parquet", index=False)
    star_all.to_parquet(combined / "ravel_star_qc_all.parquet", index=False)

    with (combined / "ravel_failures_all.jsonl").open("w", encoding="utf-8") as handle:
        for failure in failures:
            handle.write(json.dumps(failure, sort_keys=True) + "\n")

    summary = {
        "n_batches_done": sum(row.get("status") == "done" for row in batch_rows),
        "n_epoch_rows": int(len(epoch_all)),
        "n_star_rows": int(len(star_all)),
        "n_failure_rows": len(failures),
    }
    (combined / "production_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return summary

# summary = combine_completed_batches(production_root, allow_incomplete=True)
# summary

### 9. Output contract

For each batch, keep these outputs:

```text
batch_XXXXX/
  batch_XXXXX.RUNNING or batch_XXXXX.DONE
  ravel_epoch_qc_batch_XXXXX.parquet
  ravel_star_qc_batch_XXXXX.parquet
  ravel_failures_batch_XXXXX.jsonl
  completed_stars_batch_XXXXX.jsonl
  provenance_batch_XXXXX.json
  logs/
    batch.stdout.log
    batch.stderr.log
    failures/
```

The star table should store one row per star, with the full-family RV summary as the primary result and blue/red/Na as diagnostics. The epoch table should store one row per star, family, and epoch, including RVs, uncertainties, coverage flags, dropped lines, lines used, and descriptive flags. Defer final variability or multiplicity classes until the full-sample behaviour has been inspected.

### Practical defaults from the SDSS run

- Run a smoke sample first with the exact production settings.
- Use 10,000-star batches for a 100,000+ star sample.
- Use `n_workers=48` and `n_cpus_per_worker=4` on Node-07-like nodes, adjusting downward if memory pressure appears.
- Set BLAS/OpenMP thread counts to 1.
- Turn plots and corner plots off.
- Keep compact failure information and provenance, not successful per-star scratch trees.
- Record `minato` commit SHA, runner hash, environment variables, hostname, timings, row counts, and maximum worker RSS in every batch provenance file.